# Notebook 4: Motif Utility and Robustness

This notebook tests whether clean motif flows are useful for improving predictions on hard examples and corrupted inputs.

It compares five models:
- backbone baseline
- backbone + frozen full-motif hybrid
- backbone + frozen top-motif hybrid
- backbone + joint full-motif hybrid
- backbone + joint top-motif hybrid


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/jacobposchl/model-interpretability.git"
REPO_DIR = Path("/content/model_interpretability")

if REPO_DIR.parent.exists() and "google.colab" in str(get_ipython()):
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
    from google.colab import drive
    drive.mount("/content/drive")


In [ ]:
from pathlib import Path
import json
import torch

from flow_circuits.data import build_cifar10_splits
from flow_circuits.evaluation import (
    run_motif_clean_utility_experiment,
    run_motif_corruption_utility_experiment,
)
from flow_circuits.training import load_components_from_checkpoint, load_yaml_config


## Parameters


In [ ]:
CONFIG_NAME = "resnet18_z_workflow"
CONFIG_PATH = Path("configs/flow/resnet18_aligned.yaml")
HARD_PAIRS_TOP_K = 3
TRIGGER_MODE = "hard_pair_top2_and_low_margin"
MARGIN_QUANTILE = 0.25
TOP_MOTIF_FRACTION = 0.25
MIN_TOP_MOTIFS = 5
MAX_TOP_MOTIFS = 20
FIT_MAX_IMAGES = 4000
VAL_MAX_IMAGES = 1000
TEST_MAX_IMAGES = 1000
CORRUPTION_NAMES = ["gaussian_noise", "gaussian_blur", "contrast"]
CORRUPTION_SEVERITIES = [1, 3, 5]
FORCE_RERUN = False

DRIVE_ROOT = Path("/content/drive/MyDrive/flow_circuits") if Path("/content/drive/MyDrive").exists() else Path("artifacts/flow_circuits")
NB03_OUTPUT_DIR = DRIVE_ROOT / "notebook_runs" / "nb03_z_motif_discovery_and_analysis" / CONFIG_NAME
FROZEN_MOTIF_ARTIFACT = NB03_OUTPUT_DIR / "frozen" / "frozen_clean_motifs.json"
JOINT_MOTIF_ARTIFACT = NB03_OUTPUT_DIR / "joint" / "joint_clean_motifs.json"
OUTPUT_DIR = DRIVE_ROOT / "notebook_runs" / "nb04_motif_utility_and_robustness" / CONFIG_NAME


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
base_config = load_yaml_config(CONFIG_PATH)
loaders = build_cifar10_splits(
    data_dir=base_config["data"]["data_dir"],
    batch_size=base_config["data"]["batch_size"],
    num_workers=base_config["data"].get("num_workers", 4),
    seed=base_config["data"].get("seed", 0),
    augment_fit=base_config["data"].get("augment_fit", True),
    download=base_config["data"].get("download", True),
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
motif_artifacts = {
    "frozen": json.loads(FROZEN_MOTIF_ARTIFACT.read_text(encoding="utf-8")),
    "joint": json.loads(JOINT_MOTIF_ARTIFACT.read_text(encoding="utf-8")),
}

def _load_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))

def _run_or_load(path: Path, fn):
    if path.exists() and not FORCE_RERUN:
        return _load_json(path)
    result = fn(path)
    path.write_text(json.dumps(result, indent=2), encoding="utf-8")
    return result

CLEAN_RESULTS = {}
for tag, motif_artifact in motif_artifacts.items():
    checkpoint_path = motif_artifact["metadata"]["checkpoint_path"]
    components = load_components_from_checkpoint(checkpoint_path, device)
    branch_dir = OUTPUT_DIR / tag
    branch_dir.mkdir(parents=True, exist_ok=True)
    CLEAN_RESULTS[tag] = _run_or_load(
        branch_dir / "clean_utility.json",
        lambda cache_path, tag=tag, components=components, motif_artifact=motif_artifact: run_motif_clean_utility_experiment(
            components,
            motif_artifact,
            loaders["fit"],
            loaders["val"],
            loaders["test"],
            device=device,
            checkpoint_tag=tag,
            fit_max_images=FIT_MAX_IMAGES,
            val_max_images=VAL_MAX_IMAGES,
            test_max_images=TEST_MAX_IMAGES,
            top_pairs=HARD_PAIRS_TOP_K,
            trigger_mode=TRIGGER_MODE,
            margin_quantile=MARGIN_QUANTILE,
            top_motif_fraction=TOP_MOTIF_FRACTION,
            min_top_motifs=MIN_TOP_MOTIFS,
            max_top_motifs=MAX_TOP_MOTIFS,
            output_path=cache_path,
        ),
    )


In [ ]:
CORRUPTION_RESULTS = {}
for tag, motif_artifact in motif_artifacts.items():
    checkpoint_path = motif_artifact["metadata"]["checkpoint_path"]
    components = load_components_from_checkpoint(checkpoint_path, device)
    branch_dir = OUTPUT_DIR / tag
    CORRUPTION_RESULTS[tag] = _run_or_load(
        branch_dir / "corruption_utility.json",
        lambda cache_path, tag=tag, components=components, motif_artifact=motif_artifact: run_motif_corruption_utility_experiment(
            components,
            motif_artifact,
            device=device,
            checkpoint_tag=tag,
            data_dir=base_config["data"]["data_dir"],
            batch_size=base_config["data"]["batch_size"],
            corruption_names=CORRUPTION_NAMES,
            severities=CORRUPTION_SEVERITIES,
            fit_max_images=FIT_MAX_IMAGES,
            val_max_images=VAL_MAX_IMAGES,
            test_max_images=TEST_MAX_IMAGES,
            top_pairs=HARD_PAIRS_TOP_K,
            trigger_mode=TRIGGER_MODE,
            margin_quantile=MARGIN_QUANTILE,
            top_motif_fraction=TOP_MOTIF_FRACTION,
            min_top_motifs=MIN_TOP_MOTIFS,
            max_top_motifs=MAX_TOP_MOTIFS,
            num_workers=base_config["data"].get("num_workers", 4),
            seed=base_config["data"].get("seed", 0),
            augment_fit=base_config["data"].get("augment_fit", True),
            download=base_config["data"].get("download", True),
            output_path=cache_path,
        ),
    )


In [ ]:
print("Clean utility:")
for tag in ("frozen", "joint"):
    summary = CLEAN_RESULTS[tag]["summary"]
    print(f"[{tag}] backbone={summary['backbone_overall_accuracy']:.4f} | full_motif={summary['full_motif_overall_accuracy']:.4f} | top_motif={summary['top_motif_overall_accuracy']:.4f} | trigger_acc={summary['top_motif_trigger_accuracy']:.4f}")
print("\nCorruption utility:")
for tag in ("frozen", "joint"):
    summary = CORRUPTION_RESULTS[tag]["summary"]
    print(f"[{tag}] mean_backbone={summary['mean_backbone_overall_accuracy']:.4f} | mean_full_motif={summary['mean_full_motif_overall_accuracy']:.4f} | mean_top_motif={summary['mean_top_motif_overall_accuracy']:.4f}")

if CORRUPTION_RESULTS["joint"]["summary"]["mean_top_motif_gain_vs_backbone"] > CORRUPTION_RESULTS["frozen"]["summary"]["mean_top_motif_gain_vs_backbone"]:
    print("\nInterpretation: the joint branch is more useful under corruption for compact motif-based correction.")
else:
    print("\nInterpretation: the frozen branch remains competitive or better for compact motif-based correction.")
